# Notebook 03 - Constraints + Rules + Metrics（决策质量中枢）

这一节讲项目里最有价值的部分：
- `constraints`：哪些红线不能碰
- `rules`：在什么条件下改变出流策略
- `metrics`：结果好不好，怎么量化

你可以把它理解为“自动驾驶的大脑规则层 + 评分层”。


## 1) 快速准备一个场景

我们用 `SimulationService` 生成一段结果，再分别送给约束校核和评估服务。


In [ ]:
from datetime import datetime, timedelta

from pyresops.domain.reservoir import ReservoirSpec, ReservoirState, LevelStorageCurve, DischargeCapacity
from pyresops.domain.program import DispatchProgram, TimeHorizon, ModuleInstance
from pyresops.domain.forecast import ForecastBundle, ForecastSeries
from pyresops.services import ProgramService, SimulationService, EvaluationService
from pyresops.domain.constraint import Constraint, ConstraintSet

spec = ReservoirSpec(
    id='nb3_res',
    name='Notebook 3 Reservoir',
    dead_level=150.0,
    normal_level=175.0,
    flood_limit_level=155.0,
    design_flood_level=178.0,
    check_flood_level=182.0,
    total_capacity=39.3,
    flood_capacity=22.15,
    level_storage_curve=LevelStorageCurve(
        levels=[135, 145, 155, 165, 175, 185],
        storages=[0, 10, 20, 30, 39.3, 51.6],
    ),
    discharge_capacity=DischargeCapacity(
        levels=[135, 145, 155, 165, 175, 185],
        max_discharges=[0, 5000, 10000, 15000, 20000, 30000],
    ),
)

start = datetime(2024, 7, 2, 0, 0, 0)
state0 = ReservoirState(timestamp=start, level=161.0, storage=25.8, inflow=7000.0, outflow=7000.0)
forecast = ForecastBundle(
    forecast_time=start,
    series=[
        ForecastSeries(
            variable='inflow',
            timestamps=[start + timedelta(hours=i) for i in range(24)],
            values=[7000 + (i * 400 if i <= 10 else max(0, 4000 - (i-10)*300)) for i in range(24)],
        )
    ],
)

program = DispatchProgram(
    id='nb3_prog',
    name='for constraints and metrics',
    time_horizon=TimeHorizon(start=start, end=start + timedelta(hours=23), time_step=3600),
    module_sequence=[
        ModuleInstance(module_type='constant_release', parameters={'target_flow': 8000.0})
    ],
)

ps = ProgramService()
sim_s = SimulationService(spec, ps.get_module_registry())
result = sim_s.run_simulation(program, state0, forecast)
print('Simulation done, snapshots =', len(result.snapshots))


## 2) 约束校核：`constraints` 的价值

约束层负责回答：**这个方案有没有违规**。

常见约束示例：
- `level_max`: 水位不能超过某阈值
- `flow_max`: 下泄不能超过下游可承受值
- `water_supply`: 平均供水不能低于需求


In [ ]:
from pyresops.core import ConstraintValidator

constraint_set = ConstraintSet(constraints=[
    Constraint(
        id='c_level_max',
        name='max level under design flood level',
        constraint_type='level_max',
        parameters={'max_level': spec.design_flood_level},
        scope='global',
    ),
    Constraint(
        id='c_flow_max',
        name='max outflow limit',
        constraint_type='flow_max',
        parameters={'max_flow': 20000.0},
        scope='both',
    ),
    Constraint(
        id='c_supply',
        name='minimum water supply',
        constraint_type='water_supply',
        parameters={'demand': 6000.0},
        scope='global',
    ),
])

validator = ConstraintValidator(constraint_set)
violations = validator.validate_simulation(result)

print('Global violations:', len(violations))
for v in violations:
    print(v)


## 3) 策略规则：`rules` + `core.orchestrator`

规则层回答：**在当前状态下，出流应该怎么调**。

我们构造一个简化 `PolicyBundle`：
- 当入流大于阈值，就把目标下泄设高
- 再用 `clamp_outflow` 卡住上下界
- 最后仍会经过约束层和水力学能力校核


In [ ]:
from pyresops.domain.policy import PolicyBundle
from pyresops.domain.rule import RuleSet, DispatchRule, RuleAction
from pyresops.core import DecisionOrchestrator

policy = PolicyBundle(
    constraints=constraint_set,
    rules=RuleSet(rules=[
        DispatchRule(
            id='r_high_inflow',
            name='high inflow raise release',
            condition={'op': 'gt', 'left': 'inflow', 'right': 9000},
            actions=[RuleAction(action_type='set_target_outflow', parameters={'value': 12000})],
            priority=200,
        ),
        DispatchRule(
            id='r_safety_clamp',
            name='safety clamp',
            condition={'op': 'all', 'items': []},
            actions=[RuleAction(action_type='clamp_outflow', parameters={'min': 5000, 'max': 18000})],
            priority=100,
        ),
    ]),
    directives={'operator': 'notebook_demo'},
)

orchestrator = DecisionOrchestrator()
result_with_policy = sim_s.run_simulation(program, state0, forecast, policy_bundle=policy, orchestrator=orchestrator)

trace_steps = result_with_policy.metadata.get('decision_trace', [])
print('Decision trace steps:', len(trace_steps))
if trace_steps:
    print('One trace sample keys:', list(trace_steps[0].keys()))
    print('First step rule hits:', trace_steps[0].get('rule_hits'))


## 4) 评分体系：`metrics` + `EvaluationService`

评估层回答：**结果到底好不好**。

内置指标包含：
- flood（防洪）
- supply（供水）
- power（发电代理）
- ecology（生态）
- compliance（合规）


In [ ]:
ev_s = EvaluationService(spec)
evaluation = ev_s.evaluate(
    result_with_policy,
    constraint_set=constraint_set,
    include_step_scores=True,
    proxy_options={
        'env_min_flow': 4000.0,
        'max_ramp_rate': 3000.0,
        'tailwater_level': 150.0,
    },
)

print('overall_score =', round(evaluation.overall_score, 2))
print('flood_control_score =', round(evaluation.flood_control_score, 2))
print('water_supply_score =', round(evaluation.water_supply_score, 2))
print('power_generation_score =', round(evaluation.power_generation_score, 2))
print('ecological_score =', round(evaluation.ecological_score, 2))
print('extra metrics =', evaluation.additional_scores)
print('step scores length =', len(evaluation.step_scores))


## 5) 可扩展性：SPI 的意义

`constraints/rules/metrics` 三层都做成了 SPI（可插拔接口），所以：
- 不改核心引擎，也能加新约束
- 不改服务层，也能加新规则
- 不改评估框架，也能加新指标

示例模板在：`examples/extension_templates.py`


In [ ]:
from examples.extension_templates import register_demo_extensions
from pyresops.constraints import ConstraintRegistry
from pyresops.rules import RuleRegistry
from pyresops.metrics import MetricRegistry

c_registry = ConstraintRegistry()
r_registry = RuleRegistry()
m_registry = MetricRegistry()

register_demo_extensions(
    constraint_registry=c_registry,
    rule_registry=r_registry,
    metric_registry=m_registry,
)

print('Custom constraint registered? ', 'demo_outflow_ceiling' in c_registry._factories)
print('Custom rule registered?       ', 'demo_rule' in r_registry._factories)
print('Custom metric registered?     ', 'demo_stability' in m_registry._factories)


## 6) 小结

到这里你应该能回答：
- 为什么项目不把所有逻辑写死在引擎里（为了扩展）
- 为什么要分约束/规则/指标（关注点分离）
- 为什么最终会有 decision trace（可解释 + 可审计）

下一节 `notebook_04_end_to_end_mcp_and_rolling_ops.ipynb` 会把 `tools + services + storage` 串成完整业务闭环。
